# Hospital Operations & Patient Analytics
## Module 2: Data Cleaning & Transformation

This notebook performs data cleaning, transformation, validation, and normalization of the hospital patient dataset.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler


In [ ]:
# Import raw hospital dataset
df = pd.read_csv("hospital_raw_data.csv")

print("Dataset imported successfully!")
df.head()


In [ ]:
# Dataset overview
print("Number of Rows:", df.shape[0])
print("Number of Columns:", df.shape[1])
df.info()


In [ ]:
# Check missing values
missing_values = df.isnull().sum()
print("Missing values in each column:")
print(missing_values)
print("\nTotal missing values:", df.isnull().sum().sum())


In [ ]:
# Check and remove duplicate records
print("Duplicate records before cleaning:", df.duplicated().sum())
df = df.drop_duplicates()
print("Duplicate records after cleaning:", df.duplicated().sum())


In [ ]:
# Standardize text/categorical columns
text_columns = [
    "chief_complaint", "department", "attending_physician",
    "patient_gender", "insurance_type", "disposition",
    "arrival_dayofweek", "age_group"
]
for col in text_columns:
    df[col] = df[col].str.strip()
print("Text standardization completed.")


In [ ]:
# Convert arrival datetime to proper datetime format
df["arrival_datetime"] = pd.to_datetime(
    df["arrival_datetime"], errors="coerce"
)
print("Data type:", df["arrival_datetime"].dtype)


In [ ]:
# Create total waiting time
df["total_wait_time_min"] = (
    df["wait_time_triage_min"] + df["wait_time_doctor_min"]
)
df[[
    "wait_time_triage_min",
    "wait_time_doctor_min",
    "total_wait_time_min"
]].head()


In [ ]:
# Create length-of-stay category
df["stay_category"] = pd.cut(
    df["length_of_stay_hrs"],
    bins=[0, 4, 12, 24, float("inf")],
    labels=["Short", "Moderate", "Long", "Very Long"]
)
df[["length_of_stay_hrs", "stay_category"]].head()


In [ ]:
# Create billing category
df["billing_category"] = pd.cut(
    df["billed_amount_usd"],
    bins=[0, 500, 2000, 5000, float("inf")],
    labels=["Low", "Medium", "High", "Very High"]
)
df[["billed_amount_usd", "billing_category"]].head()


In [ ]:
# Create waiting-time category
df["wait_time_category"] = pd.cut(
    df["total_wait_time_min"],
    bins=[0, 30, 60, 120, float("inf")],
    labels=["Low", "Moderate", "High", "Very High"]
)
df[["total_wait_time_min", "wait_time_category"]].head()


In [ ]:
# Normalize total waiting time using Min-Max normalization
scaler = MinMaxScaler()
df["wait_time_normalized"] = scaler.fit_transform(
    df[["total_wait_time_min"]]
).flatten()
df[["total_wait_time_min", "wait_time_normalized"]].head(10)


In [ ]:
# Validate normalization
print("Minimum normalized value:", df["wait_time_normalized"].min())
print("Maximum normalized value:", df["wait_time_normalized"].max())


In [ ]:
# Final validation
print("----- Final Dataset Validation -----")
print("Total Rows:", df.shape[0])
print("Total Columns:", df.shape[1])
print("Total Missing Values:", df.isnull().sum().sum())
print("Total Duplicate Rows:", df.duplicated().sum())


In [ ]:
# Check final columns
df.columns.tolist()


In [ ]:
# Export final cleaned and transformed dataset
df.to_csv("hospital_cleaned.csv", index=False)
print("Final dataset exported successfully!")
